# 🌐 Cross-Lingual Sentiment Analysis - Google Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arif481/CrossLingual-Sentiment/blob/main/notebooks/colab_run.ipynb)

This notebook demonstrates the complete pipeline for cross-lingual sentiment analysis using multilingual transformers.

**Features:**
- Binary sentiment classification (positive/negative)
- Support for English and Bengali
- Zero-shot cross-lingual transfer
- Easy model deployment to Hugging Face Hub

---

## 🔧 Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install required packages
!pip install -q transformers datasets huggingface_hub scikit-learn pandas numpy matplotlib accelerate

In [ ]:
# Clone the repository (optional - for full functionality)
# !git clone https://github.com/arif481/CrossLingual-Sentiment.git
# %cd crosslingual-sentiment

# Or work directly in Colab with the code below

## 📊 Load Demo Data

In [ ]:
import pandas as pd

# Create demo datasets
demo_en = pd.DataFrame({
    "text": [
        "This movie is absolutely fantastic!",
        "I loved every moment of it.",
        "The acting was brilliant.",
        "What a wonderful experience.",
        "Best film I've seen this year.",
        "This was terrible and boring.",
        "I wasted my time watching this.",
        "Poor acting and weak plot.",
        "Absolutely disappointing.",
        "Worst movie I've ever seen.",
        "Great soundtrack and visuals.",
        "Heartwarming family movie.",
        "Generic and forgettable.",
        "Complete waste of talent.",
        "A true cinematic gem."
    ],
    "label": [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1]
})

demo_bn = pd.DataFrame({
    "text": [
        "এই সিনেমাটি অসাধারণ ছিল!",
        "আমি প্রতিটি মুহূর্ত উপভোগ করেছি।",
        "অভিনয় চমৎকার ছিল।",
        "কি সুন্দর অভিজ্ঞতা!",
        "এই বছরের সেরা সিনেমা।",
        "এটা একদম বাজে ছিল।",
        "সময় নষ্ট করলাম।",
        "দুর্বল অভিনয় এবং গল্প।",
        "সম্পূর্ণ হতাশাজনক।",
        "সবচেয়ে খারাপ সিনেমা।",
        "চমৎকার সাউন্ডট্র্যাক।",
        "হৃদয়গ্রাহী পারিবারিক সিনেমা।",
        "জেনেরিক এবং ভুলে যাওয়ার মতো।",
        "প্রতিভার সম্পূর্ণ অপচয়।",
        "একটি সত্যিকারের রত্ন।"
    ],
    "label": [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1]
})

print(f"English demo data: {len(demo_en)} samples")
print(f"Bengali demo data: {len(demo_bn)} samples")

display(demo_en.head())
display(demo_bn.head())

## 🔄 Data Preprocessing

In [ ]:
import re
from transformers import AutoTokenizer
from datasets import Dataset

def clean_text(text):
    """Clean text for sentiment analysis."""
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load tokenizer
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer: {MODEL_NAME}")

# Clean and tokenize
def prepare_dataset(df, tokenizer, max_length=128):
    """Prepare dataset for training."""
    df = df.copy()
    df["text"] = df["text"].apply(clean_text)
    
    dataset = Dataset.from_pandas(df)
    
    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            padding="max_length"
        )
    
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized = tokenized.rename_column("label", "labels")
    
    return tokenized

# Prepare datasets
train_en = prepare_dataset(demo_en, tokenizer)
train_bn = prepare_dataset(demo_bn, tokenizer)

print(f"\nEnglish dataset: {train_en}")
print(f"Bengali dataset: {train_bn}")

## 🤖 Model Training (Demo - 1 Epoch)

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Set seed for reproducibility
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )
    
    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1}
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training arguments (demo mode - quick training)
training_args = TrainingArguments(
    output_dir="./demo_checkpoint",
    num_train_epochs=1,  # Just 1 epoch for demo
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    report_to="none",  # Disable wandb/tensorboard for demo
)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_en,
    eval_dataset=train_bn,  # Evaluate on Bengali (zero-shot)
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Train!
print("Starting training...")
print("Training on: English")
print("Evaluating on: Bengali (zero-shot)")
print()

train_result = trainer.train()

print("\n✓ Training complete!")
print(f"Training loss: {train_result.training_loss:.4f}")

## 📈 Evaluation

In [ ]:
# Evaluate on both languages
print("Evaluating on English:")
en_results = trainer.evaluate(train_en)
print(f"  Accuracy: {en_results['eval_accuracy']:.2%}")
print(f"  F1 Score: {en_results['eval_f1']:.2%}")

print("\nEvaluating on Bengali (zero-shot):")
bn_results = trainer.evaluate(train_bn)
print(f"  Accuracy: {bn_results['eval_accuracy']:.2%}")
print(f"  F1 Score: {bn_results['eval_f1']:.2%}")

## 🎯 Inference Demo

In [ ]:
from transformers import pipeline

# Create inference pipeline
classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Test examples
test_texts = [
    # English
    "This is an amazing product! Highly recommended.",
    "Terrible experience. Complete waste of money.",
    # Bengali
    "এটি একটি অসাধারণ পণ্য! অত্যন্ত সুপারিশ করা হয়।",
    "ভয়ানক অভিজ্ঞতা। সম্পূর্ণ টাকার অপচয়।",
]

print("\n" + "="*60)
print("INFERENCE EXAMPLES")
print("="*60)

for text in test_texts:
    result = classifier(text)[0]
    emoji = "😀" if result["label"] == "positive" else "😞"
    print(f"\nText: {text[:50]}...")
    print(f"  → {emoji} {result['label'].upper()} (confidence: {result['score']:.2%})")

## 🚀 Push to Hugging Face Hub

In [ ]:
# Set your HF token (from Colab secrets or environment)
# Option 1: Use Colab secrets
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = None

# Option 2: Set manually (uncomment and add your token)
# HF_TOKEN = "your_huggingface_token_here"

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✓ Logged in to Hugging Face Hub")
else:
    print("⚠ No HF_TOKEN found. Set it in Colab secrets or manually.")
    print("  Get your token from: https://huggingface.co/settings/tokens")

In [ ]:
# Push model to Hub (only if logged in)
if HF_TOKEN:
    # Define your repo name
    REPO_NAME = "crosslingual-sentiment-demo"  # Change this!
    
    print(f"Pushing model to: {REPO_NAME}")
    
    # Push model and tokenizer
    model.push_to_hub(REPO_NAME)
    tokenizer.push_to_hub(REPO_NAME)
    
    print(f"\n✓ Model uploaded to: https://huggingface.co/{REPO_NAME}")
else:
    print("Skipping upload - set HF_TOKEN first")

## 📋 Summary

This demo showed:

1. ✅ Loading and preprocessing multilingual data
2. ✅ Training XLM-RoBERTa for sentiment classification
3. ✅ Zero-shot cross-lingual transfer (English → Bengali)
4. ✅ Running inference in multiple languages
5. ✅ Pushing the model to Hugging Face Hub

### Next Steps

- Train for more epochs with full datasets for better results
- Try the `combined` mode with both English and Bengali training data
- Deploy the model as a Streamlit app on Hugging Face Spaces

### Resources

- [Full Repository](https://github.com/arif481/CrossLingual-Sentiment)
- [Hugging Face Model](https://huggingface.co/arif481/crosslingual-sentiment-model)
- [Live Demo](https://huggingface.co/spaces/arif481/crosslingual-demo)